In [ ]:

pip install pymupdf pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 54.3 MB/s eta 0:00:00


In [ ]:
import fitz
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [8]:
from google.colab import files

uploaded = files.upload()

Saving 2020-Fundamentals-DE-NL.pdf to 2020-Fundamentals-DE-NL.pdf


In [ ]:


uploaded_file = list(uploaded.keys())[0]
doc = fitz.open(uploaded_file)

VARIANCE_THRESHOLD = 21
MIN_SIZE = 110
MAX_SIZE = 131

def colour_variance(crop):
    return float(np.std(crop.astype(float)))

def overlap(a, b, threshold=0.5):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ix = max(0, min(ax+aw, bx+bw) - max(ax, bx))
    iy = max(0, min(ay+ah, by+bh) - max(ay, by))
    inter = ix * iy
    smaller = min(aw * ah, bw * bh)
    return inter / smaller > threshold if smaller > 0 else False

def detect_swatches(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, 30, 100)
    kernel = np.ones((3, 3), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    accepted = []
    rejected = []

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)

        if w < 20 or h < 20:
            continue

        crop = img_array[y:y+h, x:x+w]
        variance = colour_variance(crop)

        reasons = []
        if not (MIN_SIZE <= w <= MAX_SIZE and MIN_SIZE <= h <= MAX_SIZE):
            reasons.append(f"size {w}x{h}")
        if abs(w - h) > 6:
            reasons.append(f"not square (diff={abs(w-h)})")
        if variance <= VARIANCE_THRESHOLD:
            reasons.append(f"flat var={variance:.0f}")

        if reasons:
            rejected.append((x, y, w, h, variance, ", ".join(reasons), crop))
        else:
            accepted.append((x, y, w, h, variance))

    filtered = []
    for c in sorted(accepted, key=lambda c: c[2]*c[3], reverse=True):
        if not any(overlap(c[:4], kept[:4]) for kept in filtered):
            filtered.append(c)

    return sorted(filtered, key=lambda c: (c[1], c[0])), rejected

for page_number, page in enumerate(doc, start=1):
    pix = page.get_pixmap(dpi=150)
    img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)

    accepted, rejected = detect_swatches(img_array)
    print(f"Page {page_number} — {len(accepted)} accepted, {len(rejected)} rejected")

    if accepted:
        cols = 6
        rows = -(-len(accepted) // cols)
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
        axes = axes.flatten()
        for i, (x, y, w, h, variance) in enumerate(accepted):
            axes[i].imshow(img_array[y:y+h, x:x+w])
            axes[i].axis("off")
            axes[i].set_title(f"var={variance:.0f}", fontsize=7, color="green")
        for j in range(len(accepted), len(axes)):
            axes[j].axis("off")
        plt.suptitle(f"Page {page_number} — ✅ ACCEPTED ({len(accepted)})", fontsize=10, color="green")
        plt.tight_layout()
        plt.show()

    if rejected:
        cols = 6
        rows = -(-len(rejected) // cols)
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
        axes = axes.flatten()
        for i, (x, y, w, h, variance, reason, crop) in enumerate(rejected):
            axes[i].imshow(crop)
            axes[i].axis("off")
            axes[i].set_title(f"{reason}", fontsize=6, color="red")
        for j in range(len(rejected), len(axes)):
            axes[j].axis("off")
        plt.suptitle(f"Page {page_number} — ❌ REJECTED ({len(rejected)})", fontsize=10, color="red")
        plt.tight_layout()
        plt.show()

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")

INSET = 8
original_filename = os.path.splitext(os.path.basename(uploaded_file))[0]
SAVE_DIR = f"/content/drive/MyDrive/glaze/swatches/type_0/{original_filename}"
os.makedirs(SAVE_DIR, exist_ok=True)

swatch_counter = 1

for page_number, page in enumerate(doc, start=1):
    pix = page.get_pixmap(dpi=150)
    img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)

    accepted, _ = detect_swatches(img_array)

    for x, y, w, h, variance in accepted:
        swatch_before = img_array[y:y+h, x:x+w]
        swatch_after  = img_array[y+INSET:y+h-INSET, x+INSET:x+w-INSET]

        fig, axes = plt.subplots(1, 2, figsize=(6, 3))
        axes[0].imshow(swatch_before, interpolation="nearest")
        axes[0].set_title(f"Before ({swatch_before.shape[1]}x{swatch_before.shape[0]})", fontsize=8)
        axes[0].axis("off")
        axes[1].imshow(swatch_after, interpolation="nearest")
        axes[1].set_title(f"After inset={INSET} ({swatch_after.shape[1]}x{swatch_after.shape[0]})", fontsize=8)
        axes[1].axis("off")
        plt.suptitle(f"P{page_number} | Swatch {swatch_counter}", fontsize=9)
        plt.tight_layout()
        plt.show()

        filename = f"{original_filename}_swatch{swatch_counter}.png"
        Image.fromarray(swatch_after).save(os.path.join(SAVE_DIR, filename))
        swatch_counter += 1

print(f"Saved to {SAVE_DIR}")
print(f"Total files: {len(os.listdir(SAVE_DIR))}")

Output hidden; open in https://colab.research.google.com to view.